# Model #3 Tag-Visible Detector (Colab)

Run this notebook on Colab with a GPU runtime. It uses the fine-tuned Qwen LoRA adapter on Hugging Face to answer a simple zero-shot question: whether a clothing tag, label, or price tag is visible in the image.

Before running the full pass, put these large local data files in Google Drive, for example under `MyDrive/resell_copilot_data/`:

- `vinted_clothing_combined.parquet`
- `kleinanzeigen_clothing_combined.parquet`

The split JSON files are expected to come from the GitHub repo.

## 1. Check GPU

In [ ]:
!nvidia-smi

## 2. Clone Repo

This assumes the latest Model #3 code has been pushed to GitHub. If `models/run_tag_detector.py` is missing after cloning, push the local repo changes first, then rerun this cell.

In [ ]:
![ -d /content/Advanced_ML/.git ] || git clone https://github.com/mchlkan/Advanced_ML.git /content/Advanced_ML
%cd /content/Advanced_ML
!git pull --ff-only
!test -f models/run_tag_detector.py && echo 'Model #3 script found.' || echo 'ERROR: models/run_tag_detector.py is missing. Push the latest repo changes first.'

## 3. Install Dependencies

`--load-in-4bit` needs `bitsandbytes`, and the adapter needs `peft`.

In [ ]:
!pip -q install -U transformers accelerate peft bitsandbytes huggingface_hub tqdm
!pip -q install "pandas==2.2.2" "pillow<12" "pyarrow>=16"

## 4. Hugging Face Login

You already accepted the model terms. Paste a Hugging Face access token when prompted. A read token is enough.

In [ ]:
from huggingface_hub import login
login()

## 5. Copy Data From Google Drive

Change `DATA_SOURCE_DIR` if your Drive folder is different.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DATA_SOURCE_DIR = '/content/drive/MyDrive/Resell_Copilot_data'

!mkdir -p data data/features
!cp "{DATA_SOURCE_DIR}/vinted_clothing_combined.parquet" data/vinted_clothing_combined.parquet
!cp "{DATA_SOURCE_DIR}/kleinanzeigen_clothing_combined.parquet" data/kleinanzeigen_clothing_combined.parquet
!ls -lh data/vinted_clothing_combined.parquet data/kleinanzeigen_clothing_combined.parquet data/splits/*.json

## 6. Validate Data Shape

In [ ]:
import json
import pandas as pd

vinted = pd.read_parquet('data/vinted_clothing_combined.parquet')
ka = pd.read_parquet('data/kleinanzeigen_clothing_combined.parquet')
print('vinted:', vinted.shape)
print('ka:', ka.shape)
for split in ['train', 'val', 'test']:
    ids = json.load(open(f'data/splits/{split}_ids.json'))
    print(split, len(ids))

## 7. Smoke Test On 10 Rows

This downloads the base model + adapter and runs only 10 examples. On a T4, model loading is the slow part.

In [ ]:
%cd /content/Advanced_ML
!git pull --ff-only


In [ ]:
!python models/run_tag_detector.py \
  --model-id Qwen/Qwen3-VL-4B-Instruct \
  --adapter-id Rengo33/qwen3vl4b-resell-adapter \
  --load-in-4bit \
  --output /content/tag_visible_smoke.parquet \
  --limit 10 \
  --batch-size 1

## 8. Inspect Smoke Test

In [ ]:
smoke = pd.read_parquet('/content/tag_visible_smoke.parquet')
smoke

In [ ]:
import pandas as pd
from PIL import Image
import io
import matplotlib.pyplot as plt

# Load smoke predictions
smoke = pd.read_parquet("/content/tag_visible_smoke.parquet")

# Load source data
vinted = pd.read_parquet("data/vinted_clothing_combined.parquet")
ka = pd.read_parquet("data/kleinanzeigen_clothing_combined.parquet")
df = pd.concat([vinted, ka], ignore_index=True)

# Join condition + image onto smoke rows
view = smoke.merge(
    df[["platform", "id", "condition_en", "title_en", "image"]],
    on=["platform", "id"],
    how="left",
)

def decode_image(cell):
    if isinstance(cell, dict) and cell.get("bytes") is not None:
        return Image.open(io.BytesIO(cell["bytes"])).convert("RGB")
    if isinstance(cell, dict) and cell.get("path"):
        return Image.open(cell["path"]).convert("RGB")
    if isinstance(cell, (bytes, bytearray)):
        return Image.open(io.BytesIO(cell)).convert("RGB")
    return cell

# Display images with prediction + condition
n = len(view)
cols = 2
rows = (n + cols - 1) // cols

plt.figure(figsize=(10, rows * 5))

for i, row in view.iterrows():
    img = decode_image(row["image"])

    ax = plt.subplot(rows, cols, i + 1)
    ax.imshow(img)
    ax.axis("off")

    ax.set_title(
        f"id: {row['id']}\n"
        f"condition: {row['condition_en']}\n"
        f"tag_visible: {row['tag_visible']} ({row['raw_response']})\n"
        f"{row.get('title_en', '')}",
        fontsize=10,
    )

plt.tight_layout()
plt.show()


In [ ]:
import pandas as pd
from pathlib import Path

# Load full data
vinted = pd.read_parquet("data/vinted_clothing_combined.parquet")
ka = pd.read_parquet("data/kleinanzeigen_clothing_combined.parquet")
df = pd.concat([vinted, ka], ignore_index=True)

# Filter New with tags
nwt = df[df["condition_en"].eq("New with tags")].copy()

print("New with tags rows:", len(nwt))
print(nwt[["platform", "id", "condition_en", "title_en"]].head(10).to_string())

# Save small temporary parquets.
# Put the 5 target rows into vinted-path, and an empty same-schema frame into ka-path.
Path("/content/model3_nwt_test").mkdir(exist_ok=True)
nwt.head(5).to_parquet("/content/model3_nwt_test/vinted_nwt_5.parquet", index=False)
df.head(0).to_parquet("/content/model3_nwt_test/ka_empty.parquet", index=False)


In [ ]:
!python models/run_tag_detector.py \
  --model-id Qwen/Qwen3-VL-4B-Instruct \
  --adapter-id Rengo33/qwen3vl4b-resell-adapter \
  --load-in-4bit \
  --vinted-path /content/model3_nwt_test/vinted_nwt_5.parquet \
  --ka-path /content/model3_nwt_test/ka_empty.parquet \
  --output /content/tag_visible_nwt_5.parquet \
  --batch-size 1

In [ ]:
pd.read_parquet("/content/tag_visible_nwt_5.parquet")

In [ ]:
import io
from PIL import Image
import matplotlib.pyplot as plt

pred = pd.read_parquet("/content/tag_visible_nwt_5.parquet")
src = pd.read_parquet("/content/model3_nwt_test/vinted_nwt_5.parquet")

view = pred.merge(
    src[["platform", "id", "condition_en", "title_en", "image"]],
    on=["platform", "id"],
    how="left",
)

def decode_image(cell):
    if isinstance(cell, dict) and cell.get("bytes") is not None:
        return Image.open(io.BytesIO(cell["bytes"])).convert("RGB")
    if isinstance(cell, dict) and cell.get("path"):
        return Image.open(cell["path"]).convert("RGB")
    if isinstance(cell, (bytes, bytearray)):
        return Image.open(io.BytesIO(cell)).convert("RGB")
    return cell

plt.figure(figsize=(12, 14))

for i, row in view.iterrows():
    ax = plt.subplot(len(view), 1, i + 1)
    ax.imshow(decode_image(row["image"]))
    ax.axis("off")
    ax.set_title(
        f"{row['platform']} | {row['id']} | condition: {row['condition_en']} | "
        f"tag_visible: {row['tag_visible']} ({row['raw_response']})\n"
        f"{row['title_en']}",
        fontsize=10,
    )

plt.tight_layout()
plt.show()

If `tag_visible` is mostly `0` or `1` and `raw_response` is mostly `yes` or `no`, run the full dataset below.

## 9. Run Full Dataset

In [ ]:
!python models/run_tag_detector.py \
  --model-id Qwen/Qwen3-VL-4B-Instruct \
  --adapter-id Rengo33/qwen3vl4b-resell-adapter \
  --load-in-4bit \
  --output data/features/tag_visible_combined.parquet \
  --batch-size 1

**Note: In Colab (free) this takes >40h to run. This is not important for now, try to rerun on runpod in the end"

## 10. Validate And Save Output

In [ ]:
out = pd.read_parquet('data/features/tag_visible_combined.parquet')
print(out.shape)
print(out['tag_visible'].value_counts(dropna=False))
print('duplicates:', out.duplicated(['platform', 'id']).sum())
print(out.head().to_string())

# Keep a copy in Drive so it survives Colab runtime reset.
!cp data/features/tag_visible_combined.parquet "{DATA_SOURCE_DIR}/tag_visible_combined.parquet"
print('Copied to:', DATA_SOURCE_DIR + '/tag_visible_combined.parquet')

## 11. Optional Download To Local Machine

In [ ]:
from google.colab import files
files.download('data/features/tag_visible_combined.parquet')